<a href="https://colab.research.google.com/github/mihirmaurya31/XAI-for-Phishing-URL-detection/blob/main/models_phish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Data Pre-processing Phase**



In [ ]:
import pandas as pd

#Read only a small sample to infer dtypes ----
sample_df = pd.read_csv("final_features.csv", nrows=10)

#Display column datatypes ----
print("=== COLUMN DATA TYPES ===")
#The output below shows the datatypes for all 115 columns
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(sample_df.dtypes)
print("\nTotal columns:", len(sample_df.columns))


#the first few rows ----
print("\n=== FIRST 5 ROWS ===")
print(sample_df.head())


=== COLUMN DATA TYPES ===
url                            object
Label                            bool
qty_dot_url                     int64
qty_hyphen_url                  int64
qty_underline_url               int64
qty_slash_url                   int64
qty_questionmark_url            int64
qty_equal_url                   int64
qty_at_url                      int64
qty_and_url                     int64
qty_exclamation_url             int64
qty_space_url                   int64
qty_tilde_url                   int64
qty_comma_url                   int64
qty_plus_url                    int64
qty_asterisk_url                int64
qty_hashtag_url                 int64
qty_dollar_url                  int64
qty_percent_url                 int64
length_url                      int64
qty_dot_domain                  int64
qty_hyphen_domain               int64
qty_underline_domain            int64
qty_slash_domain                int64
qty_questionmark_domain         int64
qty_equal_domain        

**Finding Missing Values and numerical features**

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

file_path = "final_features.csv"

# --- BASIC METADATA ---
# Read a small sample to infer columns and datatypes
sample = pd.read_csv(file_path, nrows=5000)
print("\n=== BASIC INFO ===")
print(f"Columns: {len(sample.columns)}")
print(f"Sample shape: {sample.shape}")
print("\nColumn names:\n", list(sample.columns))

# --- MISSING VALUE ANALYSIS (CHUNKED) ---
chunk_size = 200000
na_counts = pd.Series(0, index=sample.columns)
row_count = 0

for chunk in tqdm(pd.read_csv(file_path, chunksize=chunk_size)):
    na_counts += chunk.isna().sum()
    row_count += len(chunk)

missing_report = (na_counts / row_count * 100).sort_values(ascending=False)
missing_report = missing_report.reset_index()
missing_report.columns = ["Column", "Missing_%"]

print("\n=== MISSING VALUE REPORT ===")
print(missing_report.head(15))  # show top 15 columns with missing data

# Save full missing-value report if needed
missing_report.to_csv("missing_value_report.csv", index=False)

#  --- DESCRIPTIVE STATS FOR NUMERICAL FEATURES ---
num_cols = sample.select_dtypes(include=[np.number]).columns.tolist()
desc = sample[num_cols].describe().T
desc["missing_%"] = missing_report.set_index("Column").reindex(num_cols)["Missing_%"].fillna(0)
print("\n=== NUMERICAL FEATURE SUMMARY (sample) ===")
print(desc.head(10))

# --- CLASS BALANCE (Label Column) ---
if "label" in sample.columns:
    # Read only the label column to avoid memory issues
    label_counts = pd.Series(dtype=int)
    for chunk in pd.read_csv(file_path, usecols=["label"], chunksize=chunk_size):
        label_counts = label_counts.add(chunk["label"].value_counts(), fill_value=0)
    print("\n=== CLASS BALANCE ===")
    total = label_counts.sum()
    for lbl, cnt in label_counts.items():
        print(f"Label {int(lbl)}: {cnt} ({cnt/total:.2%})")

#  --- INTERESTING DISTRIBUTIONS (Light Preview) ---
interesting_cols = [c for c in ["url_length", "qty_dot_url", "qty_hyphen_url", "domain_entropy"] if c in sample.columns]

if interesting_cols:
    print("\n=== SAMPLE DISTRIBUTIONS ===")
    for col in interesting_cols:
        print(f"\n{col}:")
        print(sample[col].describe())



=== BASIC INFO ===
Columns: 115
Sample shape: (5000, 115)

Column names:
 ['url', 'Label', 'qty_dot_url', 'qty_hyphen_url', 'qty_underline_url', 'qty_slash_url', 'qty_questionmark_url', 'qty_equal_url', 'qty_at_url', 'qty_and_url', 'qty_exclamation_url', 'qty_space_url', 'qty_tilde_url', 'qty_comma_url', 'qty_plus_url', 'qty_asterisk_url', 'qty_hashtag_url', 'qty_dollar_url', 'qty_percent_url', 'length_url', 'qty_dot_domain', 'qty_hyphen_domain', 'qty_underline_domain', 'qty_slash_domain', 'qty_questionmark_domain', 'qty_equal_domain', 'qty_at_domain', 'qty_and_domain', 'qty_exclamation_domain', 'qty_space_domain', 'qty_tilde_domain', 'qty_comma_domain', 'qty_plus_domain', 'qty_asterisk_domain', 'qty_hashtag_domain', 'qty_dollar_domain', 'qty_percent_domain', 'domain_length', 'qty_dot_directory', 'qty_hyphen_directory', 'qty_underline_directory', 'qty_slash_directory', 'qty_questionmark_directory', 'qty_equal_directory', 'qty_at_directory', 'qty_and_directory', 'qty_exclamation_direct

8it [00:11,  1.44s/it]


=== MISSING VALUE REPORT ===
                  Column  Missing_%
0          server_header  47.357102
1          time_response  13.007371
2            http_status  13.007371
3         qty_hyphen_url   0.000000
4      qty_underline_url   0.000000
5          qty_slash_url   0.000000
6   qty_questionmark_url   0.000000
7          qty_equal_url   0.000000
8             qty_at_url   0.000000
9            qty_and_url   0.000000
10   qty_exclamation_url   0.000000
11         qty_space_url   0.000000
12         qty_tilde_url   0.000000
13         qty_comma_url   0.000000
14          qty_plus_url   0.000000

=== NUMERICAL FEATURE SUMMARY (sample) ===
                       count    mean       std  min  25%  50%  75%   max  \
qty_dot_url           5000.0  3.3394  2.504129  1.0  2.0  3.0  4.0  37.0   
qty_hyphen_url        5000.0  0.7330  1.436149  0.0  0.0  0.0  1.0  18.0   
qty_underline_url     5000.0  0.2584  0.736028  0.0  0.0  0.0  0.0  10.0   
qty_slash_url         5000.0  2.8812  2.312262

**Eliminating the missing values**

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

input_path = "final_features.csv"
output_path = "features_clean.csv"

chunk_size = 200000
chunks = []

# Pass 1: infer column types from a small sample
sample = pd.read_csv(input_path, nrows=5000)
numeric_cols = sample.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = sample.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Numeric cols: {len(numeric_cols)}, Categorical cols: {len(categorical_cols)}")

# Compute medians for numeric columns
numeric_medians = sample[numeric_cols].median()

# Process in chunks
for chunk in tqdm(pd.read_csv(input_path, chunksize=chunk_size)):
    # Numeric imputation
    for col in numeric_cols:
        if col in chunk.columns:
            chunk[col] = chunk[col].fillna(numeric_medians.get(col, 0))
    # Categorical imputation
    for col in categorical_cols:
        if col in chunk.columns:
            chunk[col] = chunk[col].fillna("Unknown")
    chunks.append(chunk)

# Combine and write to new file
clean_df = pd.concat(chunks, ignore_index=True)
clean_df.to_csv(output_path, index=False)

# Verify no missing values
print("\n Data cleaning complete!")
print(clean_df.isna().sum().sum(), "missing values remaining.")


Numeric cols: 112, Categorical cols: 3


8it [00:14,  1.80s/it]



 Data cleaning complete!
0 missing values remaining.


**Baseline Model Building on complete dataset**

In [ ]:
# %% [setup]
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report
)

# If xgboost isn't installed in your Colab, uncomment next line:
!pip install -q xgboost
from xgboost import XGBClassifier

RANDOM_STATE = 42
CSV_PATH = "features_clean.csv"
LABEL_COL = "Label"
TEST_SIZE = 0.20
N_SPLITS = 5

# %% [load + sanity]
df = pd.read_csv(CSV_PATH)

if LABEL_COL not in df.columns:
    raise ValueError(f"'{LABEL_COL}' not found in {CSV_PATH}.")

# Ensure boolean label (True/False); coerce common cases
if df[LABEL_COL].dtype == "object":
    df[LABEL_COL] = df[LABEL_COL].map({"True": True, "False": False, "true": True, "false": False})
if not pd.api.types.is_bool_dtype(df[LABEL_COL]):
    # if ints (0/1), convert to bool; else fail fast
    if pd.api.types.is_integer_dtype(df[LABEL_COL]) or pd.api.types.is_float_dtype(df[LABEL_COL]):
        df[LABEL_COL] = df[LABEL_COL].astype(bool)
    else:
        raise TypeError(f"Expected boolean target; got {df[LABEL_COL].dtype}")

# Optional: downcast numerics to save RAM
num_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols_all:
    if pd.api.types.is_float_dtype(df[c]):
        df[c] = pd.to_numeric(df[c], downcast="float")
    elif pd.api.types.is_integer_dtype(df[c]):
        df[c] = pd.to_numeric(df[c], downcast="integer")

X = df.drop(columns=[LABEL_COL])
y = df[LABEL_COL].values   # boolean target (sklearn treats True=1, False=0)

# Identify feature types
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

print(f"Label: {LABEL_COL} (bool). Num: {len(num_cols)} | Cat: {len(cat_cols)} | Rows: {len(df)}")

# %% [split]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# Class balance for weighting
pos = np.sum(y_train)            # True = phishing (1)
neg = len(y_train) - pos
scale_pos_weight = (neg / pos) if pos > 0 else 1.0
print(f"Train class balance → pos: {pos}, neg: {neg}, scale_pos_weight: {scale_pos_weight:.3f}")

# %% [preprocessing]
def make_ohe(sparse=True, min_freq=10):
    # compatible with different sklearn versions
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse, min_frequency=min_freq)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse)

# LR: scale numerics + sparse OHE for categoricals
preprocess_lr = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", make_ohe(sparse=True, min_freq=10), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# Trees (DT/RF/XGB): Ordinal encode categoricals, passthrough numerics
preprocess_tree = ColumnTransformer(
    [
        ("num", "passthrough", num_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# %% [models]
lr = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    class_weight="balanced",   # helps if slight imbalance
    random_state=RANDOM_STATE,
)

dt = DecisionTreeClassifier(
    class_weight="balanced",
    max_depth=None,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",        # fast & memory friendly; use "gpu_hist" if GPU available
    reg_lambda=1.0,
    reg_alpha=0.0,
    scale_pos_weight=scale_pos_weight,  # handle imbalance
    random_state=RANDOM_STATE,
    eval_metric="auc",
)

pipe_lr = Pipeline([("prep", preprocess_lr), ("clf", lr)])
pipe_dt = Pipeline([("prep", preprocess_tree), ("clf", dt)])
pipe_rf = Pipeline([("prep", preprocess_tree), ("clf", rf)])
pipe_xgb = Pipeline([("prep", preprocess_tree), ("clf", xgb)])

models = {
    "LogisticRegression": pipe_lr,
    "DecisionTree": pipe_dt,
    "RandomForest": pipe_rf,
    "XGBoost": pipe_xgb,
}

# %% [cv + test evaluation]
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

def evaluate_model(name, pipe):
    # CV
    cv_res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    cv_mean = {f"cv_{k.replace('test_', '')}": float(np.mean(v)) for k, v in cv_res.items() if k.startswith("test_")}
    print(f"\n[{name}] 5-fold CV (means): " + ", ".join([f"{k}:{v:.4f}" for k,v in cv_mean.items()]))

    # Fit on full train, evaluate on test
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)

    test_metrics = {
        "test_accuracy": accuracy_score(y_test, preds),
        "test_precision": precision_score(y_test, preds, zero_division=0),
        "test_recall": recall_score(y_test, preds),
        "test_f1": f1_score(y_test, preds),
        "test_roc_auc": roc_auc_score(y_test, proba),
        "test_pr_auc": average_precision_score(y_test, proba),
    }
    print(f"[{name}] Test: " + ", ".join([f"{k}:{v:.4f}" for k,v in test_metrics.items()]))

    # Optional: print confusion & report once per model (comment out if too verbose)
    cm = confusion_matrix(y_test, preds)
    print(f"Confusion [TN FP; FN TP]:\n{cm}")
    print(classification_report(y_test, preds, digits=4))

    return {**cv_mean, **test_metrics}

rows = []
for name, pipe in models.items():
    metrics = evaluate_model(name, pipe)
    rows.append({"model": name, **metrics})

summary = pd.DataFrame(rows).sort_values("test_pr_auc", ascending=False)
summary_path = Path("baseline_4models_metrics.csv")
summary.to_csv(summary_path, index=False)
summary
print(f"\nSaved comparison table to: {summary_path.resolve()}")


Label: Label (bool). Num: 112 | Cat: 2 | Rows: 1549360
Train class balance → pos: 125149, neg: 1114339, scale_pos_weight: 8.904

[LogisticRegression] 5-fold CV (means): cv_accuracy:0.7627, cv_precision:0.2967, cv_recall:0.9854, cv_f1:0.4561, cv_roc_auc:0.9141, cv_pr_auc:0.6210
[LogisticRegression] Test: test_accuracy:0.7620, test_precision:0.2957, test_recall:0.9820, test_f1:0.4545, test_roc_auc:0.9030, test_pr_auc:0.5886
Confusion [TN FP; FN TP]:
[[205396  73189]
 [   562  30725]]
              precision    recall  f1-score   support

       False     0.9973    0.7373    0.8478    278585
        True     0.2957    0.9820    0.4545     31287

    accuracy                         0.7620    309872
   macro avg     0.6465    0.8597    0.6512    309872
weighted avg     0.9264    0.7620    0.8081    309872


[DecisionTree] 5-fold CV (means): cv_accuracy:0.8987, cv_precision:0.4993, cv_recall:0.9067, cv_f1:0.6439, cv_roc_auc:0.9259, cv_pr_auc:0.6374
[DecisionTree] Test: test_accuracy:0.8995,

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



[RandomForest] 5-fold CV (means): cv_accuracy:0.9482, cv_precision:0.6707, cv_recall:0.9575, cv_f1:0.7888, cv_roc_auc:0.9890, cv_pr_auc:0.9211
[RandomForest] Test: test_accuracy:0.9489, test_precision:0.6729, test_recall:0.9617, test_f1:0.7918, test_roc_auc:0.9899, test_pr_auc:0.9233
Confusion [TN FP; FN TP]:
[[263959  14626]
 [  1199  30088]]
              precision    recall  f1-score   support

       False     0.9955    0.9475    0.9709    278585
        True     0.6729    0.9617    0.7918     31287

    accuracy                         0.9489    309872
   macro avg     0.8342    0.9546    0.8813    309872
weighted avg     0.9629    0.9489    0.9528    309872


[XGBoost] 5-fold CV (means): cv_accuracy:0.9176, cv_precision:0.5518, cv_recall:0.9789, cv_f1:0.7057, cv_roc_auc:0.9865, cv_pr_auc:0.8969
[XGBoost] Test: test_accuracy:0.9157, test_precision:0.5460, test_recall:0.9805, test_f1:0.7014, test_roc_auc:0.9863, test_pr_auc:0.8949
Confusion [TN FP; FN TP]:
[[253074  25511]
 [   61

**Implementing Recursive Feature Elimination technique**

In [ ]:
# %% RFE -> Reduced Features -> Retrain 4 Models (robust sampling + NaN-safe)
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report
)
from xgboost import XGBClassifier

# ------------------- CONFIG -------------------
RANDOM_STATE = 42
CSV_PATH = "features_clean.csv"
LABEL_COL = "Label"          # boolean True/False
TEST_SIZE = 0.20
N_SPLITS = 5                 # lower to 3 if RAM is tight
RFE_SAMPLE = 300_000         # size of stratified sample for RFE
MIN_PER_CLASS = 10_000       # enforce minimum observations per class in the RFE sample
KEEP_FRACTION = 0.30         # keep top 30% transformed features (min 30)
RFE_STEP = 0.20              # drop 20% each RFE iteration
# ----------------------------------------------

# ---------- Load data ----------
df = pd.read_csv(CSV_PATH)

if LABEL_COL not in df.columns:
    raise ValueError(f"Label column '{LABEL_COL}' not found in {CSV_PATH}.")

# Ensure boolean label: True/False
if df[LABEL_COL].dtype == "object":
    df[LABEL_COL] = df[LABEL_COL].map({"True": True, "False": False, "true": True, "false": False})
if not pd.api.types.is_bool_dtype(df[LABEL_COL]):
    if pd.api.types.is_integer_dtype(df[LABEL_COL]) or pd.api.types.is_float_dtype(df[LABEL_COL]):
        df[LABEL_COL] = df[LABEL_COL].astype(bool)
    else:
        raise TypeError(f"Expected boolean target; got {df[LABEL_COL].dtype}")

# Optional: downcast numerics to save RAM
for c in df.select_dtypes(include=[np.number]).columns:
    if pd.api.types.is_float_dtype(df[c]): df[c] = pd.to_numeric(df[c], downcast="float")
    else: df[c] = pd.to_numeric(df[c], downcast="integer")

X_full = df.drop(columns=[LABEL_COL])
y_full = df[LABEL_COL].values

num_cols = X_full.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_full.columns if c not in num_cols]

print(f"Initial features → Num: {len(num_cols)}, Cat: {len(cat_cols)}, Total: {len(X_full.columns)} | Rows: {len(df)}")

# ---------- Train/Test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=TEST_SIZE, stratify=y_full, random_state=RANDOM_STATE
)

# ---------- Robust stratified sample for RFE (enforces min per class) ----------
def robust_stratified_sample(X, y, total=300_000, min_per_class=5_000, random_state=42):
    y_series = pd.Series(y)
    classes = y_series.unique()
    if len(classes) < 2:
        raise ValueError(f"Need at least 2 classes; got {classes}")

    counts = y_series.value_counts()
    props = counts / counts.sum()

    desired = (props * total).round().astype(int)
    for c in counts.index:
        desired[c] = max(min_per_class, desired[c])
        desired[c] = min(desired[c], counts[c])

    desired_total = desired.sum()
    if desired_total > len(y_series):
        # reduce from largest classes until fits
        over = desired_total - len(y_series)
        order = desired.sort_values(ascending=False).index.tolist()
        i = 0
        while over > 0:
            c = order[i % len(order)]
            if desired[c] > min_per_class:
                desired[c] -= 1
                over -= 1
            i += 1

    rng = np.random.RandomState(random_state)
    idxs = []
    for c in counts.index:
        cls_idx = y_series.index[y_series == c].to_numpy()
        take = int(desired[c])
        if take > 0:
            pick = rng.choice(cls_idx, size=take, replace=False)
            idxs.append(pick)
    idxs = np.concatenate(idxs)
    Xs = X.iloc[idxs].copy()
    ys = y_series.iloc[idxs].to_numpy()

    vc = pd.Series(ys).value_counts().to_dict()
    print(f"RFE sample counts → {vc} (total={len(ys)})")
    return Xs, ys

X_rfe, y_rfe = robust_stratified_sample(
    X_train, y_train, total=RFE_SAMPLE, min_per_class=MIN_PER_CLASS, random_state=RANDOM_STATE
)

# ---------- Preprocessing for RFE (NaN-safe) ----------
prep_rfe = ColumnTransformer(
    [
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale",  StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("enc",    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

Xrfe_mat = prep_rfe.fit_transform(X_rfe)
feat_names_after = list(prep_rfe.get_feature_names_out())
p = Xrfe_mat.shape[1]

keep_k = max(30, int(np.ceil(KEEP_FRACTION * p)))
print(f"RFE will keep top {keep_k} of {p} transformed features.")

# ---------- RFE with LogisticRegression(saga) ----------
rfe_est = LogisticRegression(
    solver="saga",
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

# Final guard against single-class edge case
pos_rfe = int(np.sum(y_rfe))
neg_rfe = len(y_rfe) - pos_rfe
if pos_rfe == 0 or neg_rfe == 0:
    raise RuntimeError(f"RFE y has a single class after sampling: pos={pos_rfe}, neg={neg_rfe}")

rfe = RFE(estimator=rfe_est, n_features_to_select=keep_k, step=RFE_STEP)
rfe.fit(Xrfe_mat, y_rfe)

support_mask = rfe.support_.astype(bool)
selected_transformed = [f for f, m in zip(feat_names_after, support_mask) if m]
print(f"Selected transformed features: {len(selected_transformed)}")

# OrdinalEncoder keeps 1:1 mapping; names correspond to original columns
selected_original = list(dict.fromkeys(selected_transformed))

# Save selected features
sel_txt = Path("rfe_selected_features.txt")
with open(sel_txt, "w") as f:
    for col in selected_original:
        f.write(col + "\n")
pd.DataFrame({"selected_feature": selected_original}).to_csv("rfe_selected_features.csv", index=False)
print(f"Saved selected features to: {sel_txt.resolve()} and rfe_selected_features.csv")

# ---------- Reduce FULL train/test to selected features ----------
X_train_red = X_train[selected_original].copy()
X_test_red  = X_test[selected_original].copy()

# ---------- Preprocessors for models (NaN-safe) ----------
num_red = [c for c in selected_original if c in num_cols]
cat_red = [c for c in selected_original if c in cat_cols]

preprocess_lr = ColumnTransformer(
    [
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale",  StandardScaler())
        ]), num_red),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("enc",    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_red),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocess_tree = ColumnTransformer(
    [
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median"))
        ]), num_red),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("enc",    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_red),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# ---------- Models ----------
lr = LogisticRegression(
    solver="saga",
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)
dt = DecisionTreeClassifier(
    class_weight="balanced",
    max_depth=None,
    min_samples_leaf=3,
    random_state=RANDOM_STATE
)
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)
pos = np.sum(y_train)
neg = len(y_train) - pos
scale_pos_weight = (neg / pos) if pos > 0 else 1.0

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",    # "gpu_hist" if GPU available
    reg_lambda=1.0,
    reg_alpha=0.0,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric="auc",
)

pipe_lr  = Pipeline([("prep", preprocess_lr),  ("clf", lr)])
pipe_dt  = Pipeline([("prep", preprocess_tree), ("clf", dt)])
pipe_rf  = Pipeline([("prep", preprocess_tree), ("clf", rf)])
pipe_xgb = Pipeline([("prep", preprocess_tree), ("clf", xgb)])

models = {
    "LogisticRegression_RFE": pipe_lr,
    "DecisionTree_RFE":      pipe_dt,
    "RandomForest_RFE":      pipe_rf,
    "XGBoost_RFE":           pipe_xgb,
}

# ---------- Evaluation (CV on reduced train, then test) ----------
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

def evaluate_model(name, pipe, X_tr, y_tr, X_te, y_te):
    cv_res = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    cv_mean = {f"cv_{k.replace('test_', '')}": float(np.mean(v)) for k, v in cv_res.items() if k.startswith("test_")}
    print(f"\n[{name}] RFE-Reduced 5-fold CV (means): " + ", ".join([f"{k}:{v:.4f}" for k,v in cv_mean.items()]))

    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:, 1]
    preds = (proba >= 0.5).astype(int)

    test_metrics = {
        "test_accuracy": accuracy_score(y_te, preds),
        "test_precision": precision_score(y_te, preds, zero_division=0),
        "test_recall": recall_score(y_te, preds),
        "test_f1": f1_score(y_te, preds),
        "test_roc_auc": roc_auc_score(y_te, proba),
        "test_pr_auc": average_precision_score(y_te, proba),
    }
    print(f"[{name}] RFE-Reduced Test: " + ", ".join([f"{k}:{v:.4f}" for k,v in test_metrics.items()]))
    print("Confusion [TN FP; FN TP]:\n", confusion_matrix(y_te, preds))
    print(classification_report(y_te, preds, digits=4))

    return {**cv_mean, **test_metrics}

rows = []
for name, pipe in models.items():
    metrics = evaluate_model(name, pipe, X_train_red, y_train, X_test_red, y_test)
    rows.append({"model": name, **metrics})

summary_red = pd.DataFrame(rows).sort_values("test_pr_auc", ascending=False)
out_csv = Path("rfe_reduced_4models_metrics.csv")
summary_red.to_csv(out_csv, index=False)
print(f"\nSaved RFE reduced comparison table to: {out_csv.resolve()}")


Initial features → Num: 112, Cat: 2, Total: 114 | Rows: 1362636
RFE sample counts → {False: 265559, True: 34441} (total=300000)
RFE will keep top 35 of 114 transformed features.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Selected transformed features: 35
Saved selected features to: /content/rfe_selected_features.txt and rfe_selected_features.csv

[LogisticRegression_RFE] RFE-Reduced 5-fold CV (means): cv_accuracy:0.8471, cv_precision:0.4239, cv_recall:0.9235, cv_f1:0.5811, cv_roc_auc:0.9398, cv_pr_auc:0.6828
[LogisticRegression_RFE] RFE-Reduced Test: test_accuracy:0.8472, test_precision:0.4238, test_recall:0.9201, test_f1:0.5803, test_roc_auc:0.9392, test_pr_auc:0.6801
Confusion [TN FP; FN TP]:
 [[202093  39148]
 [  2499  28788]]
              precision    recall  f1-score   support

       False     0.9878    0.8377    0.9066    241241
        True     0.4238    0.9201    0.5803     31287

    accuracy                         0.8472    272528
   macro avg     0.7058    0.8789    0.7434    272528
weighted avg     0.9230    0.8472    0.8691    272528


[DecisionTree_RFE] RFE-Reduced 5-fold CV (means): cv_accuracy:0.9475, cv_precision:0.7103, cv_recall:0.9168, cv_f1:0.8005, cv_roc_auc:0.9498, cv_pr_auc:0

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



[RandomForest_RFE] RFE-Reduced 5-fold CV (means): cv_accuracy:0.9546, cv_precision:0.7366, cv_recall:0.9407, cv_f1:0.8262, cv_roc_auc:0.9901, cv_pr_auc:0.9336
[RandomForest_RFE] RFE-Reduced Test: test_accuracy:0.9558, test_precision:0.7418, test_recall:0.9430, test_f1:0.8304, test_roc_auc:0.9906, test_pr_auc:0.9363
Confusion [TN FP; FN TP]:
 [[230970  10271]
 [  1782  29505]]
              precision    recall  f1-score   support

       False     0.9923    0.9574    0.9746    241241
        True     0.7418    0.9430    0.8304     31287

    accuracy                         0.9558    272528
   macro avg     0.8671    0.9502    0.9025    272528
weighted avg     0.9636    0.9558    0.9580    272528


[XGBoost_RFE] RFE-Reduced 5-fold CV (means): cv_accuracy:0.9357, cv_precision:0.6480, cv_recall:0.9633, cv_f1:0.7748, cv_roc_auc:0.9888, cv_pr_auc:0.9273
[XGBoost_RFE] RFE-Reduced Test: test_accuracy:0.9356, test_precision:0.6472, test_recall:0.9645, test_f1:0.7746, test_roc_auc:0.9889, test

**SelectFromModel feature reduction technique**

In [2]:
#Feature Selection via SelectFromModel(XGBoost) -> Retrain 4 Models
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ------------------- CONFIG -------------------
RANDOM_STATE = 42
CSV_PATH = "features_clean.csv"
LABEL_COL = "Label"      # boolean True/False
TEST_SIZE = 0.20
N_SPLITS = 5             # use 3 if RAM is tight
SEL_SAMPLE = 120_000     # stratified sample size for feature selection
MIN_PER_CLASS = 5_000    # enforce a minimum per class in sample
TOP_K = 40               # keep top-K most important features from XGB
# ----------------------------------------------

# ---------- Load & sanity ----------
df = pd.read_csv(CSV_PATH)
if LABEL_COL not in df.columns:
    raise ValueError(f"Label column '{LABEL_COL}' not found.")

# Ensure boolean label
if df[LABEL_COL].dtype == "object":
    df[LABEL_COL] = df[LABEL_COL].map({"True": True, "False": False, "true": True, "false": False})
if not pd.api.types.is_bool_dtype(df[LABEL_COL]):
    if pd.api.types.is_integer_dtype(df[LABEL_COL]) or pd.api.types.is_float_dtype(df[LABEL_COL]):
        df[LABEL_COL] = df[LABEL_COL].astype(bool)
    else:
        raise TypeError(f"Expected boolean target; got {df[LABEL_COL].dtype}")

# Downcast numerics to save RAM
for c in df.select_dtypes(include=[np.number]).columns:
    if pd.api.types.is_float_dtype(df[c]): df[c] = pd.to_numeric(df[c], downcast="float")
    else: df[c] = pd.to_numeric(df[c], downcast="integer")

X_full = df.drop(columns=[LABEL_COL])
y_full = df[LABEL_COL].values

num_cols = X_full.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_full.columns if c not in num_cols]
print(f"Features → Num:{len(num_cols)} | Cat:{len(cat_cols)} | Rows:{len(df)}")

# ---------- Train/Test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=TEST_SIZE, stratify=y_full, random_state=RANDOM_STATE
)

# ---------- Robust stratified sample for selection ----------
def robust_stratified_sample(X, y, total=120_000, min_per_class=5_000, random_state=42):
    y_series = pd.Series(y)
    classes = y_series.unique()
    if len(classes) < 2:
        raise ValueError("Need at least two classes in sample.")
    counts = y_series.value_counts()
    props = counts / counts.sum()

    desired = (props * total).round().astype(int)
    for c in counts.index:
        desired[c] = max(min_per_class, desired[c])
        desired[c] = min(desired[c], counts[c])

    over = desired.sum() - min(total, len(y_series))
    if over > 0:
        order = desired.sort_values(ascending=False).index.tolist()
        i = 0
        while over > 0:
            cls = order[i % len(order)]
            if desired[cls] > min_per_class:
                desired[cls] -= 1
                over -= 1
            i += 1

    rng = np.random.RandomState(random_state)
    idxs = []
    for c in counts.index:
        cls_idx = y_series.index[y_series == c].to_numpy()
        take = int(desired[c])
        if take > 0:
            pick = rng.choice(cls_idx, size=take, replace=False)
            idxs.append(pick)
    idxs = np.concatenate(idxs)
    Xs = X.iloc[idxs].copy()
    ys = y_series.iloc[idxs].to_numpy()

    vc = pd.Series(ys).value_counts().to_dict()
    print(f"Selection sample counts → {vc} (total={len(ys)})")
    return Xs, ys

X_sel, y_sel = robust_stratified_sample(
    X_train, y_train, total=SEL_SAMPLE, min_per_class=MIN_PER_CLASS, random_state=RANDOM_STATE
)

# ---------- Preprocessor for selection (NaN-safe, compact) ----------
prep_sel = ColumnTransformer(
    [
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale",  StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("enc",    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# Transform once
Xs = prep_sel.fit_transform(X_sel)
feat_names_after = list(prep_sel.get_feature_names_out())  # identical to original names (1:1 with Ordinal)

# ---------- Train a compact XGB to get feature importances ----------
pos = np.sum(y_sel)
neg = len(y_sel) - pos
scale_pos_weight = (neg / pos) if pos > 0 else 1.0

xgb_small = XGBClassifier(
    n_estimators=300, max_depth=7, learning_rate=0.07,
    subsample=0.9, colsample_bytree=0.9,
    tree_method="hist", random_state=RANDOM_STATE, eval_metric="auc",
    scale_pos_weight=scale_pos_weight
)
xgb_small.fit(Xs, y_sel)

importances = xgb_small.feature_importances_
rank = pd.DataFrame({"feature": feat_names_after, "importance": importances}).sort_values("importance", ascending=False)
selected_transformed = rank.head(TOP_K)["feature"].tolist()
print(f"SelectFromModel(XGB) → kept Top-{TOP_K} features")

# Map back to original (names are already original due to encoder choice)
selected_original = selected_transformed
pd.DataFrame({"selected_feature": selected_original}).to_csv("sfm_xgb_selected_features.csv", index=False)
print("Saved: sfm_xgb_selected_features.csv")

# ---------- Reduce FULL train/test to selected features ----------
X_train_red = X_train[selected_original].copy()
X_test_red  = X_test[selected_original].copy()

# ---------- Build NaN-safe preprocessors for final models ----------
num_red = [c for c in selected_original if c in num_cols]
cat_red = [c for c in selected_original if c in cat_cols]

preprocess_lr = ColumnTransformer(
    [
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale",  StandardScaler())
        ]), num_red),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("enc",    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_red),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocess_tree = ColumnTransformer(
    [
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median"))
        ]), num_red),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("enc",    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_red),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# ---------- Define final models ----------
lr = LogisticRegression(
    solver="saga", max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
)
dt = DecisionTreeClassifier(
    class_weight="balanced", max_depth=None, min_samples_leaf=3, random_state=RANDOM_STATE
)
rf = RandomForestClassifier(
    n_estimators=300, max_depth=None, min_samples_leaf=2, class_weight="balanced", n_jobs=-1,
    random_state=RANDOM_STATE
)
pos_full = np.sum(y_train)
neg_full = len(y_train) - pos_full
scale_pos_weight_full = (neg_full / pos_full) if pos_full > 0 else 1.0

xgb = XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", reg_lambda=1.0, reg_alpha=0.0,
    scale_pos_weight=scale_pos_weight_full, random_state=RANDOM_STATE, eval_metric="auc"
)

from sklearn.pipeline import Pipeline as skPipeline
pipe_lr  = skPipeline([("prep", preprocess_lr),  ("clf", lr)])
pipe_dt  = skPipeline([("prep", preprocess_tree), ("clf", dt)])
pipe_rf  = skPipeline([("prep", preprocess_tree), ("clf", rf)])
pipe_xgb = skPipeline([("prep", preprocess_tree), ("clf", xgb)])

models = {
    "LogisticRegression_SFMXGB": pipe_lr,
    "DecisionTree_SFMXGB":      pipe_dt,
    "RandomForest_SFMXGB":      pipe_rf,
    "XGBoost_SFMXGB":           pipe_xgb,
}

# ---------- Evaluate (CV on reduced train, then test) ----------
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

def evaluate_model(name, pipe, X_tr, y_tr, X_te, y_te):
    cv_res = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    cv_mean = {f"cv_{k.replace('test_', '')}": float(np.mean(v)) for k, v in cv_res.items() if k.startswith("test_")}
    print(f"\n[{name}] SFM-XGB 5-fold CV (means): " + ", ".join([f"{k}:{v:.4f}" for k,v in cv_mean.items()]))

    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:, 1]
    preds = (proba >= 0.5).astype(int)

    test_metrics = {
        "test_accuracy": accuracy_score(y_te, preds),
        "test_precision": precision_score(y_te, preds, zero_division=0),
        "test_recall": recall_score(y_te, preds),
        "test_f1": f1_score(y_te, preds),
        "test_roc_auc": roc_auc_score(y_te, proba),
        "test_pr_auc": average_precision_score(y_te, proba),
    }
    print(f"[{name}] SFM-XGB Test: " + ", ".join([f"{k}:{v:.4f}" for k,v in test_metrics.items()]))
    print("Confusion [TN FP; FN TP]:\n", confusion_matrix(y_te, preds))
    print(classification_report(y_te, preds, digits=4))

    return {**cv_mean, **test_metrics}

rows = []
for name, pipe in models.items():
    metrics = evaluate_model(name, pipe, X_train_red, y_train, X_test_red, y_test)
    rows.append({"model": name, **metrics})

summary = pd.DataFrame(rows).sort_values("test_pr_auc", ascending=False)
out_csv = Path("sfm_xgb_reduced_4models_metrics.csv")
summary.to_csv(out_csv, index=False)
print(f"\nSaved SFM-XGB reduced comparison table to: {out_csv.resolve()}")


Features → Num:112 | Cat:2 | Rows:799355
Selection sample counts → {False: 96516, True: 23484} (total=120000)
SelectFromModel(XGB) → kept Top-40 features
Saved: sfm_xgb_selected_features.csv

[LogisticRegression_SFMXGB] SFM-XGB 5-fold CV (means): cv_accuracy:0.8348, cv_precision:0.8041, cv_recall:0.2061, cv_f1:0.3281, cv_roc_auc:0.7804, cv_pr_auc:0.5145


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[LogisticRegression_SFMXGB] SFM-XGB Test: test_accuracy:0.8425, test_precision:0.8610, test_recall:0.2328, test_f1:0.3666, test_roc_auc:0.7933, test_pr_auc:0.5425
Confusion [TN FP; FN TP]:
 [[127408   1176]
 [ 24002   7285]]
              precision    recall  f1-score   support

       False     0.8415    0.9909    0.9101    128584
        True     0.8610    0.2328    0.3666     31287

    accuracy                         0.8425    159871
   macro avg     0.8512    0.6118    0.6383    159871
weighted avg     0.8453    0.8425    0.8037    159871


[DecisionTree_SFMXGB] SFM-XGB 5-fold CV (means): cv_accuracy:0.8092, cv_precision:0.5093, cv_recall:0.9113, cv_f1:0.6527, cv_roc_auc:0.8942, cv_pr_auc:0.6387
[DecisionTree_SFMXGB] SFM-XGB Test: test_accuracy:0.7844, test_precision:0.4739, test_recall:0.9233, test_f1:0.6264, test_roc_auc:0.9058, test_pr_auc:0.6825
Confusion [TN FP; FN TP]:
 [[96521 32063]
 [ 2400 28887]]
              precision    recall  f1-score   support

       False     0.

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



[RandomForest_SFMXGB] SFM-XGB 5-fold CV (means): cv_accuracy:0.9194, cv_precision:0.7294, cv_recall:0.9349, cv_f1:0.8194, cv_roc_auc:0.9775, cv_pr_auc:0.9241
[RandomForest_SFMXGB] SFM-XGB Test: test_accuracy:0.9200, test_precision:0.7306, test_recall:0.9364, test_f1:0.8208, test_roc_auc:0.9786, test_pr_auc:0.9273
Confusion [TN FP; FN TP]:
 [[117779  10805]
 [  1991  29296]]
              precision    recall  f1-score   support

       False     0.9834    0.9160    0.9485    128584
        True     0.7306    0.9364    0.8208     31287

    accuracy                         0.9200    159871
   macro avg     0.8570    0.9262    0.8846    159871
weighted avg     0.9339    0.9200    0.9235    159871


[XGBoost_SFMXGB] SFM-XGB 5-fold CV (means): cv_accuracy:0.8621, cv_precision:0.5903, cv_recall:0.9649, cv_f1:0.7325, cv_roc_auc:0.9703, cv_pr_auc:0.8955
[XGBoost_SFMXGB] SFM-XGB Test: test_accuracy:0.8590, test_precision:0.5850, test_recall:0.9625, test_f1:0.7277, test_roc_auc:0.9691, test_pr_